In [3]:
# Plastic Material Flow Model for the EU in 2019
# Plastic Material Flow Analysis (MFA) Model Based on the JRC Transfer Coefficient Method (Amadei et al., 2023)
# Sectoral plastic demand allocation
# Node-based MFA structure


import pandas as pd
from pathlib import Path

# =========================================================
# USER SETTINGS
# =========================================================
excel_file = "MFA_TC1.xlsx"
sheet_name = 0

#Plastic demand total (Plastic Europe)
demand_file = "Plastic_demand.xlsx"

demand_df = pd.read_excel(
    demand_file,
    sheet_name="Plastic_demand"
)
print(demand_df)

# Extract value
total_demand_mt = float(
    demand_df.loc[
        demand_df["Year"] == 2019,
        "Plastic_demand_MT"
    ].iloc[0]
)

#Plastic demand by different sectors (Plastic Europe)
sector_file = "Sector_shares_template.xlsx"
sector_df = pd.read_excel(sector_file, sheet_name="Sector_shares")
sector_df = sector_df[sector_df["Sector_code"] != "TOTAL"].copy()

sector_shares = dict(zip(
    sector_df["Sector_code"],
    sector_df["Sector_share"]
))

# =========================================================
# HELPERS
# =========================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\n", " ").replace("\r", " ").split()).strip().lower()

def find_tc_row(df, from_text, to_text):
    from_text = clean_text(from_text)
    to_text = clean_text(to_text)

    mask = (
        df["_FROM_CLEAN"].str.strip() == from_text
    ) & (
        df["_TO_CLEAN"].str.strip() == to_text
    )

    matches = df.loc[mask]

    if matches.empty:
        raise ValueError(f"No exact match for FROM='{from_text}' and TO='{to_text}'")

    if len(matches) > 1:
        print("\n⚠️ MULTIPLE EXACT MATCHES FOUND:")
        print(matches[["FROM", "TO"] + sector_cols].to_string(index=False))
        raise ValueError("Duplicate rows exist → clean your Excel file")

    return matches.iloc[0]

def to_float(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, str):
        x = x.strip().replace("%", "")
        if x == "":
            return 0.0
    return float(x)

# =========================================================
# READ EXCEL
# =========================================================
excel_path = Path(excel_file)
if not excel_path.exists():
    raise FileNotFoundError(f"Excel file not found: {excel_file}")

df = pd.read_excel(excel_path, sheet_name=sheet_name)
df.columns = [str(c).strip() for c in df.columns]

required_cols = ["FROM", "TO"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in the Excel sheet.")

sector_cols = list(sector_shares.keys())
missing_sector_cols = [c for c in sector_cols if c not in df.columns]
if missing_sector_cols:
    raise ValueError(f"Missing sector columns in Excel: {missing_sector_cols}")

df = df.dropna(subset=["FROM", "TO"]).copy()
df["_FROM_CLEAN"] = df["FROM"].apply(clean_text)
df["_TO_CLEAN"] = df["TO"].apply(clean_text)

# Read TC from JRC file
# =========================================================
# TCs: PLASTIC PRODUCTS MANUFACTURING
# =========================================================
row_plastic_to_pre_cons_waste = find_tc_row(df, "plastic products manufacturing", "pre-consumption waste")
row_plastic_to_semi = find_tc_row(df, "plastic products manufacturing", "semi-finished products manufacturing")
row_plastic_to_finished = find_tc_row(df, "plastic products manufacturing", "finished products manufacturing")
row_plastic_to_losses = find_tc_row(df, "plastic products manufacturing", "losses")

tc_plastic_to_pre_cons_waste = {s: to_float(row_plastic_to_pre_cons_waste[s]) for s in sector_cols}
tc_plastic_to_semi = {s: to_float(row_plastic_to_semi[s]) for s in sector_cols}
tc_plastic_to_finished = {s: to_float(row_plastic_to_finished[s]) for s in sector_cols}
tc_plastic_to_losses = {s: to_float(row_plastic_to_losses[s]) for s in sector_cols}

# =========================================================
# TCs: SEMI-FINISHED
# =========================================================
row_import_to_semi = find_tc_row(df, "import", "semi-finished products manufacturing")
row_semi_to_export = find_tc_row(df, "semi-finished products manufacturing", "export")
row_semi_to_finished = find_tc_row(df, "semi-finished products manufacturing", "finished products manufacturing")
row_semi_to_consumption = find_tc_row(df, "semi-finished products manufacturing", "consumption")

tc_import_to_semi = {s: to_float(row_import_to_semi[s]) for s in sector_cols}
tc_semi_to_export = {s: to_float(row_semi_to_export[s]) for s in sector_cols}
tc_semi_to_finished = {s: to_float(row_semi_to_finished[s]) for s in sector_cols}
tc_semi_to_consumption = {s: to_float(row_semi_to_consumption[s]) for s in sector_cols}

# =========================================================
# TCs: FINISHED
# =========================================================
row_import_to_finished = find_tc_row(df, "import", "finished products manufacturing")
row_finished_to_consumption = find_tc_row(df, "finished products manufacturing", "consumption")
row_finished_to_export = find_tc_row(df, "finished products manufacturing", "export")

tc_import_to_finished = {s: to_float(row_import_to_finished[s]) for s in sector_cols}
tc_finished_to_consumption = {s: to_float(row_finished_to_consumption[s]) for s in sector_cols}
tc_finished_to_export = {s: to_float(row_finished_to_export[s]) for s in sector_cols}

# =========================================================
# TCs: CONSUMPTION
# =========================================================
row_consumption_to_waste = find_tc_row(df, "consumption", "waste generation")
row_consumption_to_losses = find_tc_row(df, "consumption", "losses")
row_consumption_to_stock = find_tc_row(df, "consumption", "stock")

tc_consumption_to_waste = {s: to_float(row_consumption_to_waste[s]) for s in sector_cols}
tc_consumption_to_losses = {s: to_float(row_consumption_to_losses[s]) for s in sector_cols}
tc_consumption_to_stock = {s: to_float(row_consumption_to_stock[s]) for s in sector_cols}

# =========================================================
# TCs: WASTE GENERATION
# =========================================================
row_waste_to_export = find_tc_row(df, "waste generation", "export")
row_waste_to_mixed = find_tc_row(df, "waste generation", "mixed waste collection")
row_waste_to_separate = find_tc_row(df, "waste generation", "separate waste collection")
row_waste_to_losses = find_tc_row(df, "waste generation", "losses")
row_waste_to_mismanaged = find_tc_row(df, "waste generation", "mismanaged waste")
row_import_to_waste = find_tc_row(df, "import", "waste generation")

tc_waste_to_export = {s: to_float(row_waste_to_export[s]) for s in sector_cols}
tc_waste_to_mixed = {s: to_float(row_waste_to_mixed[s]) for s in sector_cols}
tc_waste_to_separate = {s: to_float(row_waste_to_separate[s]) for s in sector_cols}
tc_waste_to_losses = {s: to_float(row_waste_to_losses[s]) for s in sector_cols}
tc_waste_to_mismanaged = {s: to_float(row_waste_to_mismanaged[s]) for s in sector_cols}
tc_import_to_waste = {s: to_float(row_import_to_waste[s]) for s in sector_cols}

# =========================================================
# TCs: MIXED WASTE COLLECTION
# =========================================================
row_mixed_to_recycling = find_tc_row(df, "mixed waste collection", "recycling")
row_mixed_to_incineration = find_tc_row(df, "mixed waste collection", "incineration")
row_mixed_to_landfill = find_tc_row(df, "mixed waste collection", "landfill")

tc_mixed_to_recycling = {s: to_float(row_mixed_to_recycling[s]) for s in sector_cols}
tc_mixed_to_incineration = {s: to_float(row_mixed_to_incineration[s]) for s in sector_cols}
tc_mixed_to_landfill = {s: to_float(row_mixed_to_landfill[s]) for s in sector_cols}

# =========================================================
# TCs: SEPARATE WASTE COLLECTION
# =========================================================
row_separate_to_recycling = find_tc_row(df, "separate waste collection", "recycling")
row_separate_to_incineration = find_tc_row(df, "separate waste collection", "incineration")
row_separate_to_landfill = find_tc_row(df, "separate waste collection", "landfill")
row_separate_to_reuse = find_tc_row(df, "separate waste collection", "reuse")

tc_separate_to_recycling = {s: to_float(row_separate_to_recycling[s]) for s in sector_cols}
tc_separate_to_incineration = {s: to_float(row_separate_to_incineration[s]) for s in sector_cols}
tc_separate_to_landfill = {s: to_float(row_separate_to_landfill[s]) for s in sector_cols}
tc_separate_to_reuse = {s: to_float(row_separate_to_reuse[s]) for s in sector_cols}

# =========================================================
# TCs: RECYCLING
# =========================================================
row_import_to_recycling = find_tc_row(df, "import", "recycling")
row_Mismanaged_to_recycling = find_tc_row(df, "Mismanaged waste", "recycling")
row_recycling_to_recyclates = find_tc_row(df, "recycling", "recyclates")
row_recycling_to_incineration = find_tc_row(df, "recycling", "incineration")
row_recycling_to_landfill = find_tc_row(df, "recycling", "landfill")

tc_import_to_recycling = {s: to_float(row_import_to_recycling[s]) for s in sector_cols}
tc_Mismanaged_to_recycling = {s: to_float(row_Mismanaged_to_recycling[s]) for s in sector_cols}
tc_recycling_to_recyclates = {s: to_float(row_recycling_to_recyclates[s]) for s in sector_cols}
tc_recycling_to_incineration = {s: to_float(row_recycling_to_incineration[s]) for s in sector_cols}
tc_recycling_to_landfill = {s: to_float(row_recycling_to_landfill[s]) for s in sector_cols}

# =========================================================
# TCs: INCINEARTION (Incineration to landfill )
# =========================================================
row_incineration_to_losses = find_tc_row(df, "incineration", "losses")
# row_incineration_to_landfill = find_tc_row(df, "incineration", "landfill")

tc_incineration_to_losses = {s: to_float(row_incineration_to_losses[s]) for s in sector_cols}
#tc_incineration_to_landfill = {s: to_float(row_incineration_to_landfill[s]) for s in sector_cols}

# =========================================================
# TCs: LANDFILL
# =========================================================
row_landfill_to_losses = find_tc_row(df, "landfill", "losses")
tc_landfill_to_losses = {s: to_float(row_landfill_to_losses[s]) for s in sector_cols}

# TCs: Losses
row_Mismanaged_to_losses = find_tc_row(df, "Mismanaged waste", "losses")
tc_Mismanaged_to_losses = {s: to_float(row_Mismanaged_to_losses[s]) for s in sector_cols}

# =========================================================
# TCs: RECYCLATES
# =========================================================
row_recyclates_to_export = find_tc_row(df, "recyclates", "export")
row_recyclates_to_plastic = find_tc_row(df, "recyclates", "plastic products manufacturing")

tc_recyclates_to_export = {s: to_float(row_recyclates_to_export[s]) for s in sector_cols}
tc_recyclates_to_plastic = {s: to_float(row_recyclates_to_plastic[s]) for s in sector_cols}


# =========================================================
# CALCULATIONS
# =========================================================
rows = []

for s in sector_cols:
    share = sector_shares[s]
    D_s = share * total_demand_mt

    # -----------------------------------------------------
    # 1. Plastic products manufacturing node
    # -----------------------------------------------------
    pre_dom = tc_plastic_to_pre_cons_waste[s] * D_s
    Semi_dom = tc_plastic_to_semi[s] * D_s
    Final_dom = tc_plastic_to_finished[s] * D_s
    loss_dom = tc_plastic_to_losses[s] * D_s

    plastic_outputs_sum = pre_dom + Semi_dom + Final_dom + loss_dom
    balance_diff_plastic_pellets = D_s - plastic_outputs_sum

    # -----------------------------------------------------
    # 2. Semi-finished node
    # -----------------------------------------------------
    I_semi = tc_import_to_semi[s] * Semi_dom
    E_semi = tc_semi_to_export[s] * Semi_dom

    X_semi_eff = Semi_dom + I_semi - E_semi
    F_semi_to_fin = tc_semi_to_finished[s] * X_semi_eff
    C_semi = tc_semi_to_consumption[s] * X_semi_eff

    Semi_finished_inputs = Semi_dom + I_semi
    Semi_finished_outputs = C_semi + E_semi + F_semi_to_fin
    balance_diff_Semi = Semi_finished_inputs - Semi_finished_outputs

    # -----------------------------------------------------
    # 3. Finished node
    # -----------------------------------------------------
    I_fin = tc_import_to_finished[s] * Final_dom
    E_fin = tc_finished_to_export[s] * Final_dom

    C_fin = tc_finished_to_consumption[s] * (Final_dom + I_fin - E_fin)

    finished_inputs = Final_dom + I_fin
    finished_outputs = C_fin + E_fin
    balance_diff_fin = finished_inputs - finished_outputs

    # -----------------------------------------------------
    # 4. Consumption node
    # -----------------------------------------------------
    Consumption_total = C_semi + C_fin

    Waste_gen = tc_consumption_to_waste[s] * Consumption_total
    Cons_losses = tc_consumption_to_losses[s] * Consumption_total
    Stock = tc_consumption_to_stock[s] * Consumption_total

    Consumption_outputs = Waste_gen + Cons_losses + Stock
    balance_diff_consumption = Consumption_total - Consumption_outputs

    # -----------------------------------------------------
    # 5. Waste generation node
    # -----------------------------------------------------
    Waste_import = tc_import_to_waste[s] * Waste_gen
    Waste_generation_inputs = Waste_gen + Waste_import
    Waste_export = tc_waste_to_export[s] * Waste_gen
    waste_tc_sum = (
    tc_waste_to_mixed[s]
    + tc_waste_to_separate[s]
    + tc_waste_to_losses[s]
    + tc_waste_to_mismanaged[s]
)
    Waste_mixed = tc_waste_to_mixed[s]/ waste_tc_sum  * (Waste_generation_inputs - Waste_export)
    Waste_separate = tc_waste_to_separate[s]/ waste_tc_sum* (Waste_generation_inputs - Waste_export)
    Waste_losses = tc_waste_to_losses[s] / waste_tc_sum * (Waste_generation_inputs - Waste_export)
    Waste_mismanaged = tc_waste_to_mismanaged[s]/ waste_tc_sum * (Waste_generation_inputs - Waste_export)
    
    Waste_generation_outputs = (
        Waste_export
        + Waste_mixed
        + Waste_separate
        + Waste_losses
        + Waste_mismanaged
    )
    balance_diff_waste_generation = Waste_generation_inputs - Waste_generation_outputs

    # -----------------------------------------------------
    # 6. Mixed waste collection node
    # -----------------------------------------------------
    Mixed_input = Waste_mixed
    Mixed_to_recycling = tc_mixed_to_recycling[s] * Mixed_input
    Mixed_to_incineration = tc_mixed_to_incineration[s] * Mixed_input
    Mixed_to_landfill = tc_mixed_to_landfill[s] * Mixed_input

    Mixed_outputs = Mixed_to_recycling + Mixed_to_incineration + Mixed_to_landfill
    balance_diff_mixed = Mixed_input - Mixed_outputs

    # -----------------------------------------------------
    # 7. Separate waste collection node
    # -----------------------------------------------------
    Separate_input = Waste_separate + pre_dom
    
    Separate_to_recycling = tc_separate_to_recycling[s]* Separate_input
    Separate_to_incineration = tc_separate_to_incineration[s] * Separate_input
    Separate_to_landfill = tc_separate_to_landfill[s]* Separate_input
    Separate_to_reuse    = tc_separate_to_reuse[s] * Separate_input

    Separate_outputs = Separate_to_recycling + Separate_to_incineration + Separate_to_landfill+Separate_to_reuse
    balance_diff_separate = Separate_input - Separate_outputs

    # -----------------------------------------------------
    # 8. Recycling node
    # -----------------------------------------------------
    
    Mismanaged_to_recycling=tc_Mismanaged_to_recycling[s] * Waste_mismanaged
    Import_to_recycling = tc_import_to_recycling[s] * (Mixed_to_recycling+Separate_to_recycling+Mismanaged_to_recycling)
    Recycling_input = Mixed_to_recycling + Separate_to_recycling + Import_to_recycling+Mismanaged_to_recycling

    Recycling_to_recyclates = tc_recycling_to_recyclates[s] * Recycling_input
    Recycling_to_incineration = tc_recycling_to_incineration[s] * Recycling_input
    Recycling_to_landfill = tc_recycling_to_landfill[s] * Recycling_input

    Recycling_outputs = (
        Recycling_to_recyclates
        + Recycling_to_incineration
        + Recycling_to_landfill
    )
    balance_diff_recycling = Recycling_input - Recycling_outputs

    # -----------------------------------------------------
    # 9. Incineration node
    # inputs from mixed, separate, and recycling
    # NB: Incineration_to_landfill did not consider in the paper
    # -----------------------------------------------------
    Incineration_input = (Mixed_to_incineration+ Separate_to_incineration+ Recycling_to_incineration)

    Incineration_to_losses = tc_incineration_to_losses[s] * Incineration_input
    #Incineration_to_landfill = tc_incineration_to_landfill[s] * Incineration_input

    Incineration_final_sink = Incineration_input - Incineration_to_losses
    Incineration_outputs = Incineration_to_losses + Incineration_final_sink

    balance_diff_incineration = Incineration_input - Incineration_outputs

    # -----------------------------------------------------
    # 10. Landfill node
    # Incineration to landill: skip 
    # -----------------------------------------------------
    Landfill_input = (Mixed_to_landfill+ Separate_to_landfill+ Recycling_to_landfill)

    Landfill_to_losses = tc_landfill_to_losses[s] * Landfill_input

    Landfill_final_sink = Landfill_input - Landfill_to_losses

    Landfill_outputs = Landfill_to_losses + Landfill_final_sink

    balance_diff_landfill = Landfill_input - Landfill_outputs

    # -----------------------------------------------------
    # 11. Mismanaged waste node
    # -----------------------------------------------------
    Mismanaged_input = Waste_mismanaged

    Mismanaged_to_losses=   tc_Mismanaged_to_losses[s] * Mismanaged_input

    Mismanaged_outputs = (Mismanaged_to_recycling+ Mismanaged_to_losses)

    Mismanaged_to_environment = Mismanaged_input - Mismanaged_outputs

    Mismanaged_outputs = (Mismanaged_to_recycling + Mismanaged_to_losses + Mismanaged_to_environment ) 
    
    balance_diff_mismanaged = (Mismanaged_input - Mismanaged_outputs)

    # -----------------------------------------------------
    # 12. Losses node (terminal sink)
    # -----------------------------------------------------
    loss_dom = tc_plastic_to_losses[s] * D_s
    Cons_losses = tc_consumption_to_losses[s] * Consumption_total
    Waste_losses = tc_waste_to_losses[s] * (Waste_generation_inputs - Waste_export)
    Mismanaged_to_losses=   tc_Mismanaged_to_losses[s] * Mismanaged_input
    Incineration_to_losses = tc_incineration_to_losses[s] * Incineration_input
    Landfill_to_losses = tc_landfill_to_losses[s] * Landfill_input

    Losses_input = ( loss_dom + Cons_losses + Waste_losses + Mismanaged_to_losses + Incineration_to_losses + Landfill_to_losses )

    Losses_outputs = 0.0
    balance_diff_losses = Losses_input - Losses_outputs

    # -----------------------------------------------------
    # 13.Recyclates node
    # -----------------------------------------------------
    Recyclates_input = Recycling_to_recyclates
    Recyclates_to_export = (tc_recyclates_to_export[s] * Recyclates_input)
    Recyclates_to_plastic = (tc_recyclates_to_plastic[s] * Recyclates_input)
    Recyclates_outputs = (Recyclates_to_export+ Recyclates_to_plastic)
    balance_diff_recyclates = (Recyclates_input - Recyclates_outputs)

    # -----------------------------------------------------
    # 14. Export node
    # -----------------------------------------------------
    Export_input = (E_semi + E_fin + Waste_export + Recyclates_to_export)

    Export_outputs = Export_input
    balance_diff_export = Export_input - Export_outputs

    rows.append({
        "Sector": s,
        "Share": share,
        "Domestic_input_MT": D_s,

        # Plastic products manufacturing
        "Pre_cons_waste_MT": pre_dom,
        "Plast_pellets_losses_MT": loss_dom,
        "Plastic_to_semi_MT": Semi_dom,
        "Plastic_to_finished_MT": Final_dom,
        "Plastic_pellets_inputs_MT": D_s,
        "Plastic_pellets_outputs_MT": plastic_outputs_sum,
        "Plastic_pellets_balance_diff_MT": balance_diff_plastic_pellets,

        # Semi-finished
        "Semi_dom_MT": Semi_dom,
        "Import_semi_MT": I_semi,
        "Export_semi_MT": E_semi,

        "Semi_to_finished_MT": F_semi_to_fin,
        "Semi_to_consumption_MT": C_semi,
        "Semi_Finished_inputs_MT": Semi_finished_inputs,
        "Semi_Finished_outputs_MT": Semi_finished_outputs,
        "Semi_Finished_balance_diff_MT": balance_diff_Semi,

        # Finished
        "Finished_domestic_MT": Final_dom,
        "Import_finished_MT": I_fin,
        "Export_finished_MT": E_fin,
       
        "Finished_to_consumption_MT": C_fin,
        "Finished_inputs_MT": finished_inputs,
        "Finished_outputs_MT": finished_outputs,
        "Finished_balance_diff_MT": balance_diff_fin,

        # Consumption
        "Consumption_inputs_MT": Consumption_total,
        "Consumption_to_waste_generation_MT": Waste_gen,
        "Consumption_losses_MT": Cons_losses,
        "Consumption_to_stock_MT": Stock,
        "Consumption_outputs_MT": Consumption_outputs,
        "Consumption_balance_diff_MT": balance_diff_consumption,

        # Waste generation
        "Waste_generation_inputs_MT": Waste_generation_inputs,
        "Import_to_waste_generation_MT": Waste_import,
        "Waste_to_export_MT": Waste_export,
        "Waste_to_mixed_collection_MT": Waste_mixed,
        "Waste_to_separate_collection_MT": Waste_separate,
        "Waste_losses_MT": Waste_losses,
        "Waste_to_mismanaged_MT": Waste_mismanaged,
        "Waste_generation_outputs_MT": Waste_generation_outputs,
        "Waste_generation_balance_diff_MT": balance_diff_waste_generation,

        # Mixed waste collection
        "Mixed_collection_input_MT": Mixed_input,
        "Mixed_to_recycling_MT": Mixed_to_recycling,
        "Mixed_to_incineration_MT": Mixed_to_incineration,
        "Mixed_to_landfill_MT": Mixed_to_landfill,
        "Mixed_collection_outputs_MT": Mixed_outputs,
        "Mixed_collection_balance_diff_MT": balance_diff_mixed,

        # Separate waste collection
        "Separate_collection_input_MT": Separate_input,
        "Separate_to_recycling_MT": Separate_to_recycling,
        "Separate_to_incineration_MT": Separate_to_incineration,
        "Separate_to_landfill_MT": Separate_to_landfill,
        "Separate_to_reuse_MT": Separate_to_reuse,
        "Separate_collection_outputs_MT": Separate_outputs,
        "Separate_collection_balance_diff_MT": balance_diff_separate,


        # Recycling
        "Mismanaged_to_recycling_MT": Mismanaged_to_recycling,
        "Import_to_recycling_MT": Import_to_recycling,
        "Recycling_input_MT": Recycling_input,
        "Recycling_to_recyclates_MT": Recycling_to_recyclates,
        "Recycling_to_incineration_MT": Recycling_to_incineration,
        "Recycling_to_landfill_MT": Recycling_to_landfill,
        "Recycling_outputs_MT": Recycling_outputs,
        "Recycling_balance_diff_MT": balance_diff_recycling,

         # Incineration
        "Incineration_input_MT": Incineration_input,
        "Incineration_to_losses_MT": Incineration_to_losses,
        #"Incineration_to_landfill_MT": Incineration_to_landfill,
        "Incineration_final_sink_MT": Incineration_final_sink,
        "Incineration_outputs_MT": Incineration_outputs,
        "Incineration_balance_diff_MT": balance_diff_incineration,
      
        # Mismanaged node
        "Mismanaged_input_MT": Mismanaged_input,
        "Mismanaged_to_recycling_MT": Mismanaged_to_recycling,
        "Mismanaged_to_losses_MT": Mismanaged_to_losses,
        "Mismanaged_to_environment_MT": Mismanaged_to_environment,
        "Mismanaged_outputs_MT": Mismanaged_outputs,
        "Mismanaged_balance_diff_MT": balance_diff_mismanaged,
         # Landfill
        "Landfill_input_MT": Landfill_input,
        "Landfill_to_losses_MT": Landfill_to_losses,
        "Landfill_final_sink_MT": Landfill_final_sink,
        "Landfill_outputs_MT": Landfill_outputs,
        "Landfill_balance_diff_MT": balance_diff_landfill,
        # Losses
        "Plast_pellets_losses_MT": loss_dom,
        "Mismanaged_to_losses_MT": Mismanaged_to_losses,
        "Consumption_losses_MT": Cons_losses,
        "Waste_losses_MT": Waste_losses,
        "Incineration_to_losses_MT": Incineration_to_losses,
        "Losses_input_MT": Losses_input,
        "Losses_output_MT":Losses_outputs,
        "balance_diff_losses_MT":balance_diff_losses, 

        # Recyclates node
        "Recyclates_input_MT": Recyclates_input,
        "Recyclates_to_export_MT": Recyclates_to_export,
        "Recyclates_to_plastic_products_MT": Recyclates_to_plastic,
        "Recyclates_outputs_MT": Recyclates_outputs,
        "Recyclates_balance_diff_MT": balance_diff_recyclates,

        # Export node
        "Export_input_MT": Export_input,
        "Export_outputs_MT": Export_outputs,
        "Export_balance_diff_MT": balance_diff_export,
    })

# =========================================================
# RESULTS TABLE
# =========================================================

result = pd.DataFrame(rows)

sum_cols = [c for c in result.columns if c != "Sector"]

totals_dict = {"Sector": "TOTAL"}
for c in sum_cols:
    totals_dict[c] = result[c].sum()

totals = pd.DataFrame([totals_dict])

result = pd.concat([result, totals], ignore_index=True)

# =========================================================
# REPLACE SECTOR CODES WITH FULL NAMES
# =========================================================

sector_name_map = {
    "P": "Packaging",
    "C": "Construction",
    "T": "Transport",
    "E": "EEE",
    "A": "Agriculture",
    "C&T": "Clothing and textiles",
    "H": "Healthcare",
    "F": "Fishing",
    "O": "Other",
    "TOTAL": "TOTAL"
}

result["Sector"] = result["Sector"].replace(sector_name_map)

# ---------------------------------------------------------
# VERTICAL TABLE
# ---------------------------------------------------------
result_vertical = result.set_index("Sector").T

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("\n=== MFA VERTICAL RESULTS ===\n")
print(result_vertical)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------
out_file = "Plastic_MFA.xlsx"

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    result_vertical.to_excel(writer, sheet_name="MFA_vertical")
    result.to_excel(writer, sheet_name="MFA_horizontal", index=False)

print(f"\nSaved MFA tables to: {out_file}")

   Year  Plastic_demand_MT
0  2019          53.300000

=== MFA VERTICAL RESULTS ===

Sector                      Packaging  Construction  Transport      EEE  \
Share                        0.396000      0.204000   0.096000 0.062000   
Domestic_input_MT           21.106800     10.873200   5.116800 3.304600   
Pre_cons_waste_MT            0.274388      0.000000   0.000000 0.000000   
Plast_pellets_losses_MT      0.021107      0.010873   0.005117 0.003305   
Plastic_to_semi_MT           8.611574      4.120943   1.074528 0.046264   
...                               ...           ...        ...      ...   
Recyclates_outputs_MT        3.845500      0.556860   0.386465 0.270805   
Recyclates_balance_diff_MT   0.000000      0.000000   0.000000 0.000000   
Export_input_MT              3.717622      1.963333   1.450025 0.818135   
Export_outputs_MT            3.717622      1.963333   1.450025 0.818135   
Export_balance_diff_MT       0.000000      0.000000   0.000000 0.000000   

Sector        

In [4]:
# Plastic Material Flow Analysis Sankey Diagram
# EU Plastic MFA by Sector, 2019
# Based on JRC Transfer Coefficient Methodology
# Purpose:
# This script reads the sectoral MFA results from the Excel file
# produced by the plastic MFA model and visualizes the flows using
# a Sankey diagram.

import plotly.graph_objects as go

# =========================================================
# USER SETTINGS
# =========================================================
excel_file = "Plastic_MFA.xlsx"
sheet_name = "MFA_vertical"
output_html = "MFA_Sankey_by_sector-2019.html"
# =========================================================
# READ DATA
# =========================================================
df_vertical = pd.read_excel(excel_file, sheet_name=sheet_name, index_col=0)

df = df_vertical.T.reset_index()
df = df.rename(columns={"index": "Sector"})
df = df[df["Sector"].astype(str).str.upper() != "TOTAL"].copy()
sector_name_map = {
    "P": "Packaging",
    "C": "Construction",
    "T": "Transport",
    "E": "EEE",
    "A": "Agriculture",
    "C&T": "Clothing & Textiles",
    "H": "Healthcare",
    "F": "Fishing",
    "O": "Other",
}

df["Sector_full"] = df["Sector"].map(sector_name_map).fillna(df["Sector"])

def getv(row, col, default=0.0):
    return float(row[col]) if col in row.index and pd.notna(row[col]) else default

# =========================================================
# NODES
# =========================================================
nodes = [
    "Import","Pl. to manuf.","Pre-cons. waste","Semi-fin. prod.",
    "Fin. prod.","Consum.","Stock","Waste gen.","Mism. waste",
    "Mix. waste coll.","Sep. waste coll.","Recycling","Recyclates",
    "Incineration","Landfill","Losses","Export","Reuse","Environment"
]

node_index = {n:i for i,n in enumerate(nodes)}

# =========================================================
# POSITIONS
# =========================================================
x = [0.02,0.08,0.22,0.22,0.38,0.52,0.70,0.56,0.70,
     0.70,0.70,0.84,0.94,0.84,0.90,0.97,0.98,0.60,0.94]

y = [0.05,0.25,0.38,0.55,0.25,0.35,0.08,0.58,0.42,
     0.58,0.76,0.68,0.68,0.52,0.62,0.50,0.92,0.90,0.32]

# =========================================================
# COLORS
# =========================================================
sector_colors = {
    "Packaging": "rgba(102,153,255,0.70)",
    "Construction": "rgba(255,153,102,0.70)",
    "Transport": "rgba(102,204,153,0.70)",
    "EEE": "rgba(204,153,255,0.70)",
    "Agriculture": "rgba(153,204,102,0.70)",
    "Clothing & Textiles": "rgba(255,204,102,0.70)",
    "Healthcare": "rgba(255,102,153,0.70)",
    "Fishing": "rgba(102,204,255,0.70)",
    "Other": "rgba(180,180,180,0.70)",
}

# =========================================================
# BUILD LINKS
# =========================================================
links = []

def add_link(src, tgt, val, sector):
    if pd.isna(val):
        return
    val = float(val)
    if abs(val) < 1e-12:
        return
    links.append({
        "source": node_index[src],
        "target": node_index[tgt],
        "value": val,
        "sector": sector,
        "color": sector_colors.get(sector, "rgba(150,150,150,0.5)")
    })

for _, row in df.iterrows():

    sector = row["Sector_full"]

    add_link("Pl. to manuf.","Pre-cons. waste",getv(row,"Pre_cons_waste_MT"),sector)
    add_link("Pl. to manuf.","Semi-fin. prod.",getv(row,"Plastic_to_semi_MT"),sector)
    add_link("Pl. to manuf.","Fin. prod.",getv(row,"Plastic_to_finished_MT"),sector)
    add_link("Pl. to manuf.","Losses",getv(row,"Plast_pellets_losses_MT"),sector)

    add_link("Recyclates","Pl. to manuf.",getv(row,"Recyclates_to_plastic_products_MT"),sector)

    add_link("Import","Semi-fin. prod.",getv(row,"Import_semi_MT"),sector)
    add_link("Import","Fin. prod.",getv(row,"Import_finished_MT"),sector)
    add_link("Import","Waste gen.",getv(row,"Import_to_waste_generation_MT"),sector)
    add_link("Import","Recycling",getv(row,"Import_to_recycling_MT"),sector)

    add_link("Semi-fin. prod.","Fin. prod.",getv(row,"Semi_to_finished_MT"),sector)
    add_link("Semi-fin. prod.","Consum.",getv(row,"Semi_to_consumption_MT"),sector)
    add_link("Semi-fin. prod.","Export",getv(row,"Export_semi_MT"),sector)

    add_link("Fin. prod.","Consum.",getv(row,"Finished_to_consumption_MT"),sector)
    add_link("Fin. prod.","Export",getv(row,"Export_finished_MT"),sector)

    add_link("Consum.","Waste gen.",getv(row,"Consumption_to_waste_generation_MT"),sector)
    add_link("Consum.","Stock",getv(row,"Consumption_to_stock_MT"),sector)
    add_link("Consum.","Losses",getv(row,"Consumption_losses_MT"),sector)

    add_link("Waste gen.","Export",getv(row,"Waste_to_export_MT"),sector)
    add_link("Waste gen.","Mix. waste coll.",getv(row,"Waste_to_mixed_collection_MT"),sector)
    add_link("Waste gen.","Sep. waste coll.",getv(row,"Waste_to_separate_collection_MT"),sector)
    add_link("Waste gen.","Mism. waste",getv(row,"Waste_to_mismanaged_MT"),sector)
    add_link("Waste gen.","Losses",getv(row,"Waste_losses_MT"),sector)

    add_link("Pre-cons. waste","Sep. waste coll.",getv(row,"Pre_cons_waste_MT"),sector)

    add_link("Mism. waste","Recycling",getv(row,"Mismanaged_to_recycling_MT"),sector)
    add_link("Mism. waste","Losses",getv(row,"Mismanaged_to_losses_MT"),sector)
    
    add_link("Mix. waste coll.","Recycling",getv(row,"Mixed_to_recycling_MT"),sector)
    add_link("Mix. waste coll.","Incineration",getv(row,"Mixed_to_incineration_MT"),sector)
    add_link("Mix. waste coll.","Landfill",getv(row,"Mixed_to_landfill_MT"),sector)
    add_link("Mism. waste","Environment",getv(row,"Mismanaged_to_environment_MT"),sector)

    add_link("Sep. waste coll.","Recycling",getv(row,"Separate_to_recycling_MT"),sector)
    add_link("Sep. waste coll.","Incineration",getv(row,"Separate_to_incineration_MT"),sector)
    add_link("Sep. waste coll.","Landfill",getv(row,"Separate_to_landfill_MT"),sector)
    add_link("Sep. waste coll.","Reuse",getv(row,"Separate_to_reuse_MT"),sector)

    add_link("Recycling","Recyclates",getv(row,"Recycling_to_recyclates_MT"),sector)
    add_link("Recycling","Incineration",getv(row,"Recycling_to_incineration_MT"),sector)
    add_link("Recycling","Landfill",getv(row,"Recycling_to_landfill_MT"),sector)
    add_link("Recycling","Losses",getv(row,"Recycling_to_losses_MT"),sector)

    add_link("Recyclates","Export",getv(row,"Recyclates_to_export_MT"),sector)

    add_link("Incineration","Landfill",getv(row,"Incineration_to_landfill_MT"),sector)
    add_link("Incineration","Losses",getv(row,"Incineration_to_losses_MT"),sector)

    add_link("Landfill","Losses",getv(row,"Landfill_to_losses_MT"),sector)

# =========================================================
# NODE VALUES
# =========================================================
node_in = {n:0 for n in nodes}
node_out = {n:0 for n in nodes}

for L in links:
    node_out[nodes[L["source"]]] += L["value"]
    node_in[nodes[L["target"]]] += L["value"]

plastic_input_total = df["Plastic_pellets_inputs_MT"].sum()

node_values = {}
for n in nodes:
    if n == "Import":
        node_values[n] = node_out[n]
    elif n == "Pl. to manuf.":
        node_values[n] = plastic_input_total
    else:
        node_values[n] = node_in[n]

labels = [f"{n} {node_values[n]:.2f}" for n in nodes]

# =========================================================
# MAIN SANKEY
# =========================================================
fig = go.Figure()

fig.add_trace(go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=20,
        thickness=14,
        line=dict(color="black", width=1),
        label=labels,
        x=x,
        y=y,
        color="lightgray"
    ),
    link=dict(
        source=[L["source"] for L in links],
        target=[L["target"] for L in links],
        value=[L["value"] for L in links],
        color=[L["color"] for L in links],
        customdata=[L["sector"] for L in links],
        hovertemplate="Sector: %{customdata}<br>%{value:.2f} Mt<extra></extra>"
    )
))

# =========================================================
# LEGEND (dummy traces)
# =========================================================
for sec, col in sector_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=12, color=col),
        name=sec,
        showlegend=True
    ))



# =========================================================
# LAYOUT
# =========================================================
fig.update_layout(
    title=dict(
        text="Material Flow Analysis by Sector",
        x=0.05,
        y=0.97,
        font=dict(size=20)
    ),
    width=1400,
    height=650,
    font=dict(size=11, family="Arial"),
    paper_bgcolor="white",
    plot_bgcolor="white",
xaxis=dict(
    visible=False,
    showgrid=False,
    zeroline=False
),

yaxis=dict(
    visible=False,
    showgrid=False,
    zeroline=False
),
    legend=dict(
        title=None,
        orientation="h",
        x=0.5,
        y=-0.10,
        xanchor="center",
        yanchor="top",
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        font=dict(size=11)
    ),

    margin=dict(l=20, r=180, t=50, b=90)
)
fig.write_html(output_html, include_plotlyjs=True)
fig.show()

print("Saved:", output_html)


Saved: MFA_Sankey_by_sector-2019.html
